# 🎙️ AI 智能发音私教 (AI Pronunciation Coach)

这是一款基于多模态大模型与端侧加速技术的智能发音评估与纠错应用。无论是外语口语练习还是母语朗读训练，只需输入你的练习文本并录制语音，AI 就能像真实的私人教练一样，为你提供精准、多维度的发音诊断。

### ✨ 核心功能亮点

* 🗣️ **多模态精准对比**：告别传统纯语音识别（ASR）的“过度纠错”陷阱。系统利用 **Qwen2-Audio** 多模态大模型，直接对比目标文本与你的原始音频流，精准捕捉音节发音瑕疵与中式口音。
* 🎯 **多维度细致反馈**：不仅能精准定位读错的字词并提供发音建议，还能综合评估你的语调、节奏，揪出漏读或多读的内容，并输出 0-100 的量化评分。
* ⚡ **端侧极致加速**：底层深度集成 **Intel OpenVINO™** 推理引擎，利用 INT4 模型量化技术，在普通个人电脑（CPU/iGPU）上即可实现低延迟、保护隐私的全本地化流畅运行。
* 👨‍🏫 **沉浸式双语辅导**：支持中英文双语的发音评测，系统会全程使用清晰易懂的中文为你提供详尽的指导建议，打造无障碍的学习体验。

本 Notebook 用于演示如何复用当前仓库代码完成完整流程：
1. 安装依赖
2. 下载 Qwen3-TTS 与 Qwen2-Audio OpenVINO 模型
3. 输入文本并生成标准示范音频
4. （可选）修改纠错 Prompt
5. 使用 Qwen2-Audio 对音频进行评估并输出结果
6. 启动 Gradio 交互界面

In [ ]:
%git clone https://github.com/WHITE-BLANK/modelscope-workshop
%cd modelscope-workshop
%pip install -r requirements.txt


## 下载模型

- TTS 模型：Qwen3-TTS-CustomVoice-0.6B-fp16-ov
- 纠错模型：Qwen2-Audio-7B-Instruct-Openvino-4Bit

In [ ]:
from pathlib import Path
import os
import subprocess

tts_model_dir = Path("model/Qwen3-TTS-CustomVoice-0.6B-fp16-ov")
qwen2_audio_model_dir = Path("model/qwen2-audio-7b-ov-int4")

# 1) 下载 TTS 模型
if not tts_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-TTS-CustomVoice-0.6B-fp16-ov", local_dir=str(tts_model_dir))
    print(f"模型已下载到: {tts_model_dir}")
else:
    print(f"模型已存在: {tts_model_dir}，跳过下载")

# 2) 下载 Qwen2-Audio OpenVINO INT4 模型（需安装huggingface-cli）
if not qwen2_audio_model_dir.exists():
    env = os.environ.copy()
    env["HF_ENDPOINT"] = "https://hf-mirror.com"
    cmd = [
        "huggingface-cli",
        "download",
        "wolfofbackstreet/Qwen2-Audio-7B-Instruct-Openvino-4Bit",
        "--local-dir",
        "./qwen2-audio-7b-ov-int4",
    ]
    subprocess.run(cmd, check=True, env=env)
    print(f"模型已下载到: {qwen2_audio_model_dir}")
else:
    print(f"模型已存在: {qwen2_audio_model_dir}，跳过下载")

## 加载 Pipeline


In [ ]:
from pathlib import Path
from pronunciation_correction_pipeline import PronunciationCorrectionPipeline

project_root = Path.cwd()
device = "CPU"  
speaker = "vivian"

pipeline = PronunciationCorrectionPipeline(
    project_root=str(project_root),
    tts_model_dir=str(tts_model_dir),
    qwen_audio_model_dir=str(qwen2_audio_model_dir),
    device=device,
    tts_speaker=speaker,
)

print("Pipeline 加载完成")

## 输入待朗读文本并生成标准示范音频

In [ ]:
import IPython.display as ipd

target_text = "今天天气很好，我们一起练习标准普通话发音。"
reference_audio_path = "outputs/notebook_reference.wav"

generated_path = pipeline.generate_reference_audio(
    text=target_text,
    output_wav_path=reference_audio_path,
    language="Chinese",
)

print(f"示范音频路径: {generated_path}")
ipd.display(ipd.Audio(generated_path))

## 修改纠错 Prompt

如果不修改，默认会根据目标文本自动构建 Prompt。

In [ ]:
default_prompt = (
        "你是一位严格且专业的发音教练。请听用户的录音，并将其与下方的“目标文本”进行对比，提供详细的纠错反馈。\n"
        "【重要指令】：请全程使用中文（简体）进行回答，并严格按照下方的【评分基准】进行客观打分。\n\n"
        "【评分基准】（采用满分100分的扣分制）：\n"
        "- 基础分为 100 分。\n"
        "- 严重发音错误：每一个完全读错或导致意思改变的单词，扣 5 分。\n"
        "- 轻微发音瑕疵：每一个元音/辅音发音不到位但不影响理解的单词，扣 2 分。\n"
        "- 漏读或多读：每一个漏掉或多加的单词，扣 3 分。\n"
        "- 语调与节奏：若整体重音放错、结巴或断句极其不自然，根据严重程度整体扣 5 到 10 分。\n"
        "-（注：最低得分为0分，不设负分）\n\n"
        "目标文本：\n"
        f"{original_text}\n\n"
        "请严格按照以下格式输出你的反馈：\n"
        "1. 综合评分：[填写最终分数]分。（请简要列出计算公式，例如：100 - 严重错误10分 - 漏读3分 = 87分）。\n"
        "2. 发音错误：明确指出读错的字、词，并给出正确的发音建议。\n"
        "3. 语调与节奏问题：指出重音错误、停顿不自然等现象。\n"
        "4. 漏读或多读的内容：列出具体的单词。\n"
        "5. 改进建议：一小段简短的练习指导。"
)

print("默认 Prompt:")
print(default_prompt)

# 直接修改这个字符串即可自定义评估指令
custom_prompt = default_prompt

## 使用 Qwen2-Audio 评估音频文件并输出结果

这里为了方便演示，直接用刚才生成的示范音频作为被评估音频。
实际应用中，你可以将 `user_audio_path` 换成用户上传的跟读录音。

In [ ]:
from pathlib import Path
import IPython.display as ipd

# 方式1：读取本地用户录音（推荐）
local_audio_path = Path("outputs/notebook_reference.wav")  # 修改为你的本地音频路径

if not local_audio_path.exists():
    raise FileNotFoundError(f"未找到本地音频文件: {local_audio_path}")

user_audio_path = str(local_audio_path)
print(f"将评估本地音频: {user_audio_path}")
ipd.display(ipd.Audio(user_audio_path))

# 方式2：若你想用上一步生成的示范音频进行测试，可改为：
# user_audio_path = generated_path

result = pipeline.run(
    original_text=target_text,
    user_audio_path=user_audio_path,
    reference_audio_out="outputs/notebook_reference_from_run.wav",
    language="Chinese",
    correction_prompt=custom_prompt,
 )

print("=== 最终使用的纠错 Prompt ===")
print(result.correction_prompt)
print("\n=== Qwen2-Audio 评估结果 ===")
print(result.correction_text)

## 启动交互式 Gradio 界面

In [ ]:
from pronunciation_correction_gradio import create_demo

demo = create_demo()
print("Gradio 正在启动（non-inline 模式）...")
demo.queue().launch(
    debug=True,
    inline=False,
    prevent_thread_lock=True,
    show_error=True,
    share=False,
    server_name="127.0.0.1",
    server_port=7860,
)